In [21]:
import sys
import os
import urllib3

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)

Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [22]:
import pandas as pd
from datetime import datetime, timedelta
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=2)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")

display(observed_src)


Querying owner: HTOC Org

Retrieved 768 unique observed indicators.

Retrieved 768 unique observed indicators.


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,associatedGroups.data,md5,sha1,sha256,size,text,address,Block,url,indicator
82,6755399459033758,2025-06-16T17:42:40Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-08T16:34:23Z,3.0,64.0,RITM0589984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.70.100.5
129,4809962,2024-08-08T18:17:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-08T16:16:23Z,3.0,24.0,Sharing malicious indicators flagged by Virust...,...,"[{'id': 447870, 'dateAdded': '2024-08-08T18:17...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,165.22.54.16
130,6755399459033714,2025-06-16T17:42:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-08T16:16:22Z,3.0,74.0,TASK0912447,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.42.116.213
144,6755399465229502,2025-07-28T17:33:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-08T16:13:50Z,3.0,70.0,TASK0902923,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,198.235.24.253
147,5629499561376894,2025-07-28T17:33:44Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-08T16:13:47Z,3.0,70.0,TASK0902923,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,198.235.24.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9137,6755399460007437,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:12Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,"[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.28.200.170
9169,6755399460007783,2025-07-02T12:05:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:10Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,"[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,181.174.115.9
9195,6755399460007842,2025-07-02T12:05:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:09Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,"[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.237.210.82
9233,6755399460008013,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:07Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,"[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122.54.133.163


In [23]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251008.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251007.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251006.csv']


C:\Users\jaskew\AppData\Local\Temp\ipykernel_5416\564055639.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Loaded data from 3 files.


In [24]:
import pandas as pd
from datetime import timedelta
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION & SETUP
# ═══════════════════════════════════════════════════════════════════════════════

# Time cutoffs
cutoff = pd.Timestamp.utcnow()
cutoff_naive = cutoff.tz_convert(None)

# Required columns validation
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [c for c in required_columns if c not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def process_tag_row(row, observed_src):
    """Process a single row from observed_src to extract and filter tags."""
    tags_data = row.get('tags.data')
    if not isinstance(tags_data, list):
        return None

    # Normalize the tags for the current row
    tags_df = pd.json_normalize(tags_data)

    # Ensure string and apply VA rename before filtering
    tags_df['name'] = (
        tags_df['name']
        .astype(str)
        .str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)
    )

    # Skip if all associated groups are Adversary
    if 'associatedGroups.data' in observed_src.columns:
        ag_data = row.get('associatedGroups.data')
        if isinstance(ag_data, list) and len(ag_data) > 0:
            groups_df = pd.json_normalize(ag_data)
            if 'type' in groups_df.columns and set(groups_df['type']) == {'Adversary'}:
                return None

    # All tag names (for all_tags)
    all_tags_list = tags_df['name'].tolist()

    # Filter for API tags
    api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)].copy()
    if api_tags.empty:
        return None

    # Add metadata columns
    meta_cols = [
        'summary', 'observations', 'description', 'type',
        'dateAdded', 'lastModified', 'lastObserved', 'webLink',
        'rating', 'confidence', 'id', 'associatedGroups.data'
    ]
    for col in meta_cols:
        api_tags[col] = [row.get(col)] * len(api_tags)

    # Attach all tags list
    if len(api_tags) > 0:
        api_tags['all_tags'] = [all_tags_list] * len(api_tags)

    return api_tags

def map_observed_dates(filtered_tags, observed_data_df):
    """Map observed dates from observed_data_df to filtered_tags."""
    if filtered_tags.empty:
        return filtered_tags
    
    # Clean name -> match OpDiv (remove trailing ' Splunk API')
    filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)
    filtered_tags['observed_date'] = pd.NaT
    
    # Ensure observed_data_df obs_date is datetime (naive)
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')

    for idx, r in filtered_tags.iterrows():
        summary = r.get('summary')
        cleaned_name = r.get('cleaned_name')
        if pd.isna(summary) or pd.isna(cleaned_name):
            continue
        match = observed_data_df[
            (observed_data_df['indicator'] == summary) &
            (observed_data_df['OpDiv'] == cleaned_name)
        ]
        if not match.empty:
            filtered_tags.at[idx, 'observed_date'] = match['obs_date'].iloc[0]

    filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')
    
    # Drop helper column
    filtered_tags.drop(columns=['cleaned_name'], inplace=True, errors='ignore')
    
    return filtered_tags

def apply_filters_and_grouping(filtered_tags, cutoff_naive):
    """Apply time filters, partner grouping, and other filtering criteria."""
    if filtered_tags.empty:
        return pd.DataFrame()
    
    # Last 2 days by observed_date (naive)
    recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()

    if recent_tags.empty:
        return recent_tags
    
    # Partner extraction and grouping
    recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

    partner_groups = (
        recent_tags.groupby('summary')['partner']
        .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
        .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
    )

    recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

    # Additional filters
    recent_tags = recent_tags[recent_tags['partner_count'] >= 2]

    # Deduplicate by summary
    recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

    # Drop unneeded columns if present
    cols_to_drop = [
        'techniqueId', 'tactics.data', 'tactics.count',
        'platforms.data', 'platforms.count', 'partner', 'name'
    ]
    recent_tags = recent_tags.drop(columns=[c for c in cols_to_drop if c in recent_tags.columns], errors='ignore')

    # Remove rows where all_tags contains unwanted markers
    if 'all_tags' in recent_tags.columns:
        recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'I&W' in x)]
        recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'htoc_wl' in x)]

    return recent_tags

def extract_group_ids(recent_tags):
    """Extract group IDs from associatedGroups.data."""
    if 'associatedGroups.data' in recent_tags.columns:
        recent_tags['group_id'] = recent_tags['associatedGroups.data'].apply(
            lambda x: x[0]['id'] if isinstance(x, list) and len(x) > 0 and 'id' in x[0] else None
        )
        # Convert group_id to string type and remove trailing decimals if it exists
        if 'group_id' in recent_tags.columns:
            recent_tags['group_id'] = recent_tags['group_id'].apply(
                lambda x: str(int(float(x))) if pd.notna(x) and str(x) != 'None' else x
            ).astype(str)
    return recent_tags

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PROCESSING PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

print("Starting tag processing pipeline...")

# Step 1: Process tags from observed_src
print("Processing tags from observed_src...")
all_filtered = []

for _, row in observed_src.iterrows():
    processed_tags = process_tag_row(row, observed_src)
    if processed_tags is not None:
        all_filtered.append(processed_tags)

# Step 2: Create filtered_tags DataFrame
print("Creating filtered_tags DataFrame...")
filtered_tags = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()

if not filtered_tags.empty:
    # Ensure datetime columns
    filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce', utc=True)
    filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce', utc=True)
    print(f"Created filtered_tags with {len(filtered_tags)} rows")
else:
    print("No filtered tags found")

# Step 3: Map observed dates
print("Mapping observed dates...")
filtered_tags = map_observed_dates(filtered_tags, observed_data_df)

# Step 4: Apply filters and grouping
print("Applying filters and partner grouping...")
recent_tags = apply_filters_and_grouping(filtered_tags, cutoff_naive)

# Step 5: Extract group IDs
print("Extracting group IDs...")
recent_tags = extract_group_ids(recent_tags)

# Final summary
print(f"Processing complete! Final dataset shape: {recent_tags.shape}")
if not recent_tags.empty:
    print(f"Partners represented: {recent_tags['partners'].nunique()} unique partner combinations")

display(recent_tags)

Starting tag processing pipeline...
Processing tags from observed_src...
Creating filtered_tags DataFrame...
Creating filtered_tags DataFrame...
Created filtered_tags with 5839 rows
Mapping observed dates...
Created filtered_tags with 5839 rows
Mapping observed dates...
Applying filters and partner grouping...
Extracting group IDs...
Processing complete! Final dataset shape: (518, 18)
Partners represented: 84 unique partner combinations
Applying filters and partner grouping...
Extracting group IDs...
Processing complete! Final dataset shape: (518, 18)
Partners represented: 84 unique partner combinations


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id
0,6755399459033758,RITM0589984,2025-10-07T19:11:04Z,109.70.100.5,7292,Address,2025-06-16 17:42:40+00:00,2025-10-08T16:34:23Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,NaN,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-10-08,3,"CMS, FDA, HRSA",nan
3,4809962,Sharing malicious indicators flagged by Virust...,2025-10-07T07:34:30Z,165.22.54.16,1943758,Address,2024-08-08 18:17:34+00:00,2025-10-08T16:16:23Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,24.0,"[{'id': 447870, 'dateAdded': '2024-08-08T18:17...","[Indicators, Malicious Activity, OS Splunk API...",2025-10-08,2,"CMS, HHS",447870
5,6755399459033714,TASK0912447,2025-10-07T09:09:13Z,192.42.116.213,93125,Address,2025-06-16 17:42:04+00:00,2025-10-08T16:16:22Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,74.0,NaN,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-10-07,2,"HRSA, OS",nan
7,6755399465229502,TASK0902923,2025-10-08T11:06:23Z,198.235.24.253,10377146,Address,2025-07-28 17:33:57+00:00,2025-10-08T16:13:50Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,70.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-08,8,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",nan
15,5629499561376894,TASK0902923,2025-10-08T11:06:23Z,198.235.24.66,9985302,Address,2025-07-28 17:33:44+00:00,2025-10-08T16:13:47Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,70.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-08,8,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3699,6755399460003092,TASK0912427,2025-10-08T11:06:23Z,65.49.1.209,826600,Address,2025-06-30 12:21:47+00:00,2025-10-05T01:56:40Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,74.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-07,9,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",nan
3708,6755399460003091,TASK0912427,2025-10-08T11:06:23Z,64.62.156.77,1394077,Address,2025-06-30 12:21:46+00:00,2025-10-05T01:56:40Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,74.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-08,8,"CMS, DHA, FDA, HHS, HRSA, IHS, OS, VA",nan
3716,6755399460003088,TASK0912427,2025-10-08T11:06:23Z,64.62.156.215,868481,Address,2025-06-30 12:21:44+00:00,2025-10-05T01:56:40Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,74.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-07,8,"CMS, DHA, FDA, HHS, HRSA, IHS, OS, VA",nan
3724,6755399460003076,TASK0912427,2025-10-08T11:06:23Z,65.49.1.30,1623425,Address,2025-06-30 12:21:35+00:00,2025-10-05T01:56:40Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,74.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-08,9,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, OS, VA",nan


In [25]:
import pandas as pd

# Extract unique indicators from recent_tags
indicators = recent_tags['summary'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Track indicators with no attributes
indicators_with_no_attributes = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            attributes = data.get('attributes', {}).get('data', [])

            if not attributes:
                print(f"No attributes found for indicator: {indicator}")
                # Track indicators with no attributes
                indicators_with_no_attributes.append(indicator)
            else:
                for attr in attributes:
                    attr.update({
                        'id': data.get('id'),
                        'summary': data.get('summary'),
                        'type': data.get('type'),
                        'ownerName': data.get('ownerName')
                    })
                    attributes_data.append(attr)
        # Suppress non-JSON and known 400 error responses
    except Exception as e:
        # Suppress error output, including known 400 error
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            continue
        if "Status Code: 400" in str(e):
            continue
        pass

# Create attributes 
attributes_observed_src = pd.json_normalize(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
    attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

# Drop duplicates based on 'id' to keep unique attribute records
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags for indicators that had attributes
filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]

# Filter recent_tags for indicators that had no attributes
no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

# Combine both and remove duplicates based on 'summary'
filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)
display(filtered_recent_tags)


No attributes found for indicator: 192.166.244.248
No attributes found for indicator: 37.19.207.37
No attributes found for indicator: 37.19.207.37
No attributes found for indicator: 146.70.45.166
No attributes found for indicator: 146.70.45.166
No attributes found for indicator: 37.19.200.151
No attributes found for indicator: 37.19.200.151


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id
0,4809962,Sharing malicious indicators flagged by Virust...,2025-10-07T07:34:30Z,165.22.54.16,1943758,Address,2024-08-08 18:17:34+00:00,2025-10-08T16:16:23Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,24.0,"[{'id': 447870, 'dateAdded': '2024-08-08T18:17...","[Indicators, Malicious Activity, OS Splunk API...",2025-10-08,2,"CMS, HHS",447870
1,6755399460008045,"Details\nIn late June 2025, Iran-aligned hackt...",2025-10-08T11:06:23Z,157.15.62.44,274,Address,2025-07-02 12:05:34+00:00,2025-10-08T12:46:04Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,60.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-10-07,2,"DHA, VA",5629499544001057
2,3563469,IRT 67469,2025-10-08T15:46:41Z,74.208.236.247,584,Address,2021-01-08 12:36:31+00:00,2025-10-08T07:39:41Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,75.0,"[{'id': 91700, 'dateAdded': '2020-05-06T12:47:...","[DHA Splunk API, CMS Splunk API, VA Splunk API...",2025-10-07,2,"NIH, VA",91700
3,5629499537014343,CISA observed scripted download and execution ...,2025-10-08T15:46:41Z,104.21.20.66,573,Address,2025-04-21 11:07:48+00:00,2025-10-08T07:39:38Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,51.0,NaN,"[DHA Splunk API, malicious code, VA OIS CSOC C...",2025-10-07,3,"IHS, NIH, VA",nan
4,4554012,CMS Anomali ThreatStream Indicators from 01.14...,2025-10-07T07:34:30Z,64.227.146.243,4394888,Address,2024-03-29 14:24:45+00:00,2025-10-08T07:39:38Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,20.0,"[{'id': 327567, 'dateAdded': '2024-03-29T14:24...","[cms-soc, OS Splunk API, FDA Splunk API, CMS S...",2025-10-08,2,"CMS, HHS",327567
5,4036480,RITM0589984,2025-10-07T19:11:04Z,193.189.100.195,114673,Address,2021-12-15 13:22:24+00:00,2025-10-08T07:39:36Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,"[{'id': 129288, 'dateAdded': '2021-12-15T13:04...","[OS Splunk API, FDA Splunk API, Log4j, CMS Spl...",2025-10-07,3,"CMS, FDA, HRSA",129288
6,5629499562039548,The IP address 167.94.146.20 is actively class...,2025-10-08T11:06:23Z,167.94.146.20,40285095,Address,2025-08-01 14:26:07+00:00,2025-10-08T07:13:15Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,73.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-08,8,"CDC, CMS, DHA, FDA, HRSA, IHS, OS, VA",nan
7,4890774,Sharing malicious indicators flagged by virust...,2025-10-07T07:34:30Z,162.142.125.242,182299139,Address,2024-09-13 20:48:55+00:00,2025-10-08T07:13:15Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,29.0,"[{'id': 509735, 'dateAdded': '2024-10-17T13:47...","[DHA Splunk API, Web Scanner, Reconnaissance, ...",2025-10-08,4,"CDC, CMS, HHS, NIH",509735
8,5074376,Malicious IP. Traffic attempting to exploit a ...,2025-10-07T07:34:30Z,162.142.125.247,138028166,Address,2024-11-05 21:56:21+00:00,2025-10-08T04:17:01Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,2.0,26.0,NaN,"[Malicious Activity, OS Splunk API, VA OIS CSO...",2025-10-08,2,"CMS, HHS",nan
9,2988588,RITM0589984,2025-10-07T09:09:13Z,185.220.101.21,48872,Address,2020-09-03 10:52:35+00:00,2025-10-08T03:12:48Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,"[{'id': 136273, 'dateAdded': '2022-05-10T11:58...","[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-10-08,3,"CMS, HRSA, OS",136273


In [26]:
import pandas as pd

# Path to the reported indicators file
reported_indicators_path = r"Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Reported Indicators\indicators.csv"

# Load reported indicators and filter out already-reported indicators
try:
    # Load CSV with error handling for malformed rows
    reported_indicators_df = pd.read_csv(reported_indicators_path, on_bad_lines='skip')
    print(f"Loaded reported indicators - shape: {reported_indicators_df.shape}")
    
    if not reported_indicators_df.empty:
        # Remove duplicates
        reported_indicators_df = reported_indicators_df.drop_duplicates().reset_index(drop=True)
        
        # Identify indicator column and create set for filtering
        if 'Indicator' in reported_indicators_df.columns:
            reported_set = set(reported_indicators_df['Indicator'].astype(str))
            col_name = 'Indicator'
        elif 'indicator' in reported_indicators_df.columns:
            reported_set = set(reported_indicators_df['indicator'].astype(str))
            col_name = 'indicator'
        elif 'summary' in reported_indicators_df.columns:
            reported_set = set(reported_indicators_df['summary'].astype(str))
            col_name = 'summary'
        elif len(reported_indicators_df.columns) == 1:
            col_name = reported_indicators_df.columns[0]
            reported_set = set(reported_indicators_df[col_name].astype(str))
        else:
            print("No suitable indicator column found")
            reported_set = set()
        
        print(f"Found {len(reported_set)} indicators in '{col_name}' column")
        
        # Filter out already-reported indicators from filtered_recent_tags
        if 'filtered_recent_tags' in globals() and not filtered_recent_tags.empty and reported_set:
            before_count = len(filtered_recent_tags)
            filtered_recent_tags = filtered_recent_tags[
                ~filtered_recent_tags['summary'].astype(str).isin(reported_set)
            ].reset_index(drop=True)
            after_count = len(filtered_recent_tags)
            print(f"Removed {before_count - after_count} already-reported indicators")
            print(f"Final filtered dataset: {after_count} indicators")
        elif 'filtered_recent_tags' not in globals():
            print("Warning: filtered_recent_tags not found - run previous cells first")
    else:
        print("No reported indicators loaded")
        
except Exception as e:
    print(f"Error loading reported indicators: {e}")

# Display final filtered results
if 'filtered_recent_tags' in globals():
    display(filtered_recent_tags)
else:
    print("No filtered_recent_tags to display")

Loaded reported indicators - shape: (249, 1)
Found 249 indicators in 'Indicator' column
Removed 2 already-reported indicators
Final filtered dataset: 33 indicators


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id
0,4809962,Sharing malicious indicators flagged by Virust...,2025-10-07T07:34:30Z,165.22.54.16,1943758,Address,2024-08-08 18:17:34+00:00,2025-10-08T16:16:23Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,24.0,"[{'id': 447870, 'dateAdded': '2024-08-08T18:17...","[Indicators, Malicious Activity, OS Splunk API...",2025-10-08,2,"CMS, HHS",447870
1,6755399460008045,"Details\nIn late June 2025, Iran-aligned hackt...",2025-10-08T11:06:23Z,157.15.62.44,274,Address,2025-07-02 12:05:34+00:00,2025-10-08T12:46:04Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,60.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-10-07,2,"DHA, VA",5629499544001057
2,3563469,IRT 67469,2025-10-08T15:46:41Z,74.208.236.247,584,Address,2021-01-08 12:36:31+00:00,2025-10-08T07:39:41Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,75.0,"[{'id': 91700, 'dateAdded': '2020-05-06T12:47:...","[DHA Splunk API, CMS Splunk API, VA Splunk API...",2025-10-07,2,"NIH, VA",91700
3,4554012,CMS Anomali ThreatStream Indicators from 01.14...,2025-10-07T07:34:30Z,64.227.146.243,4394888,Address,2024-03-29 14:24:45+00:00,2025-10-08T07:39:38Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,20.0,"[{'id': 327567, 'dateAdded': '2024-03-29T14:24...","[cms-soc, OS Splunk API, FDA Splunk API, CMS S...",2025-10-08,2,"CMS, HHS",327567
4,4036480,RITM0589984,2025-10-07T19:11:04Z,193.189.100.195,114673,Address,2021-12-15 13:22:24+00:00,2025-10-08T07:39:36Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,"[{'id': 129288, 'dateAdded': '2021-12-15T13:04...","[OS Splunk API, FDA Splunk API, Log4j, CMS Spl...",2025-10-07,3,"CMS, FDA, HRSA",129288
5,5629499562039548,The IP address 167.94.146.20 is actively class...,2025-10-08T11:06:23Z,167.94.146.20,40285095,Address,2025-08-01 14:26:07+00:00,2025-10-08T07:13:15Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,73.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-08,8,"CDC, CMS, DHA, FDA, HRSA, IHS, OS, VA",nan
6,4890774,Sharing malicious indicators flagged by virust...,2025-10-07T07:34:30Z,162.142.125.242,182299139,Address,2024-09-13 20:48:55+00:00,2025-10-08T07:13:15Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,29.0,"[{'id': 509735, 'dateAdded': '2024-10-17T13:47...","[DHA Splunk API, Web Scanner, Reconnaissance, ...",2025-10-08,4,"CDC, CMS, HHS, NIH",509735
7,5074376,Malicious IP. Traffic attempting to exploit a ...,2025-10-07T07:34:30Z,162.142.125.247,138028166,Address,2024-11-05 21:56:21+00:00,2025-10-08T04:17:01Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,2.0,26.0,NaN,"[Malicious Activity, OS Splunk API, VA OIS CSO...",2025-10-08,2,"CMS, HHS",nan
8,2988588,RITM0589984,2025-10-07T09:09:13Z,185.220.101.21,48872,Address,2020-09-03 10:52:35+00:00,2025-10-08T03:12:48Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,"[{'id': 136273, 'dateAdded': '2022-05-10T11:58...","[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-10-08,3,"CMS, HRSA, OS",136273
9,6755399460007636,"Details\nIn late June 2025, Iran-aligned hackt...",2025-10-08T15:46:41Z,47.251.43.115,787,Address,2025-07-02 12:05:30+00:00,2025-10-08T02:34:11Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,2.0,60.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-10-07,2,"HHS, VA",5629499544001057


In [33]:
from datetime import datetime, UTC

output_path = 'Z:\\HTOC\\HTOC Reports\\I&W Reports\\5. I&W Staging\\Spreadsheet\\Expanded'

# Generate filename with today's date in YYYYMMDD format
today_str = datetime.now(UTC).strftime("%Y%m%d")
excel_filename = f"expanded_indicators_{today_str}.xlsx"
excel_path = os.path.join(output_path, excel_filename)

# Convert all timezone-aware datetime columns to naive (remove timezone info)
for col in filtered_recent_tags.select_dtypes(include=['datetimetz']).columns:
    filtered_recent_tags[col] = filtered_recent_tags[col].dt.tz_localize(None)

# Drop the 'description' column if it exists
if 'description' in filtered_recent_tags.columns:
    filtered_recent_tags = filtered_recent_tags.drop(columns=['description'])

filtered_recent_tags.to_excel(excel_path, index=False)
print(f"Saved to {excel_path}")

Saved to Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Spreadsheet\Expanded\expanded_indicators_20251008.xlsx
